In [ ]:
# ── 0. SETUP ──────────────────────────────────────────────────────────────────
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, FloatType, ArrayType

spark = SparkSession.builder \
    .appName("MyDigitalTwin-Clustering") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Paths Docker vs local
if os.path.exists("/opt/spark/warehouse"):
    WAREHOUSE = "/opt/spark/warehouse"
else:
    # Windows local (chemin absolu)
    WAREHOUSE = os.path.abspath(os.path.join(os.getcwd(), "../../../scripts/notebooks", "..", "..", "warehouse"))
    if not os.path.exists(WAREHOUSE):
        WAREHOUSE = "C:/Users/arnau/Documents/MyDigitalTwin/warehouse"

print(f"Warehouse: {WAREHOUSE}")
assert os.path.exists(WAREHOUSE), f"Warehouse introuvable: {WAREHOUSE}"

---
## PARTIE C — Fusion → interest_profiles

On fusionne les deux types de clusters pour créer un profil enrichi par cluster de contenu : `label_contenu · profil_comportemental_dominant`.

La home page lira cette table.

In [ ]:
# ── C1. FUSION : QUEL PROFIL COMPORTEMENTAL DOMINE CHAQUE CLUSTER CONTENU ? ───
# On joint content_cluster + beh_cluster sur les mêmes events
# Pour cela, on refait l'inférence sur les données communes (all_text)

# Ajouter le cluster contenu sur all_text
content_labeled = km_content_model.transform(tfidf_df) \
    .select("text", "hour", "weekday", "platform", "weight", "content_cluster")

# Ajouter les features comportementales puis le beh_cluster sur le même DF
# On applique les mêmes transformations que partie B
fused = content_labeled \
    .withColumn("hour_sin",    F.sin(2 * 3.14159 * F.col("hour") / 24)) \
    .withColumn("hour_cos",    F.cos(2 * 3.14159 * F.col("hour") / 24)) \
    .withColumn("weekday_sin", F.sin(2 * 3.14159 * F.col("weekday") / 7)) \
    .withColumn("weekday_cos", F.cos(2 * 3.14159 * F.col("weekday") / 7)) \
    .withColumn("weight_norm", F.coalesce(F.col("weight"), F.lit(1.0)).cast("double"))

fused = indexer_model.transform(fused)
fused = encoder_model.transform(fused)
fused = assembler_beh.transform(fused)
fused = scaler_model.transform(fused)
fused = km_beh_model.transform(fused)

print("Fusion contenu + comportemental ok.")
fused.select("text", "content_cluster", "beh_cluster").show(10, truncate=50)

In [ ]:
# ── C2. CREATION DES PROFILS FUSIONNÉS ────────────────────────────────────────
# Pour chaque content_cluster → trouver le beh_cluster dominant

cross_tab = fused.groupBy("content_cluster", "beh_cluster").count()

# beh_cluster dominant par content_cluster (argmax)
from pyspark.sql.window import Window
w = Window.partitionBy("content_cluster").orderBy(F.desc("count"))
dominant_beh = cross_tab.withColumn("rank", F.rank().over(w)) \
    .filter(F.col("rank") == 1) \
    .select("content_cluster", "beh_cluster").withColumnRenamed("beh_cluster", "dominant_beh_cluster")

dominant_beh.show()

# Construire la table interest_profiles
interest_rows = []
for row in dominant_beh.collect():
    cid  = row["content_cluster"]
    bcid = row["dominant_beh_cluster"]
    
    content_info = content_cluster_info[cid]
    beh_info     = beh_cluster_info[bcid]
    
    c_label  = CONTENT_LABELS.get(cid,  {}).get("label", f"Cluster {cid}")
    b_label  = BEH_LABELS.get(bcid,    {}).get("label", f"Profil {bcid}")
    b_emoji  = BEH_LABELS.get(bcid,    {}).get("emoji", "")
    
    # Label hybride : "🎵 Musique · Nuit créative"
    hybrid_label = f"{c_label} · {beh_info['time_period']}"
    
    interest_rows.append((
        cid,
        c_label,
        b_label,
        hybrid_label,
        CONTENT_LABELS.get(cid, {}).get("emoji", "❓"),
        content_info["top_terms"][:10],
        content_info["top_platforms"],
        content_info["sample_texts"][:5],
        round(beh_info["avg_hour"], 1),
        beh_info["time_period"],
        beh_info["day_type"],
        content_info["item_count"]
    ))
    
    print(f"Profile {cid}: {hybrid_label}")
    print(f"  Keywords: {content_info['top_terms'][:8]}")

In [ ]:
# ── C3. ECRITURE interest_profiles ────────────────────────────────────────────
schema_ip = StructType([
    StructField("cluster_id",       IntegerType(), False),
    StructField("content_label",    StringType(),  False),
    StructField("behavioral_label", StringType(),  True),
    StructField("hybrid_label",     StringType(),  True),
    StructField("emoji",            StringType(),  True),
    StructField("keywords",         ArrayType(StringType()), True),
    StructField("top_platforms",    ArrayType(StringType()), True),
    StructField("sample_items",     ArrayType(StringType()), True),
    StructField("avg_hour",         DoubleType(),  True),
    StructField("time_period",      StringType(),  True),
    StructField("day_type",         StringType(),  True),
    StructField("item_count",       LongType(),    True),
])

interest_profiles_df = spark.createDataFrame(interest_rows, schema_ip)

out_path_ip = os.path.join(WAREHOUSE, "interest_profiles")
interest_profiles_df.write.mode("overwrite").parquet(out_path_ip)

print(f"Ecrit dans : {out_path_ip}")
interest_profiles_df.select("cluster_id", "hybrid_label", "keywords", "time_period", "item_count") \
    .show(truncate=60)

---
## PARTIE D — Visualisation PCA 2D (Galaxie)

Projection PCA des clusters de contenu pour visualisation.

In [ ]:
# ── D1. PCA 2D SUR LES CLUSTERS CONTENU ───────────────────────────────────────
from pyspark.ml.feature import PCA

pca = PCA(k=2, inputCol="features", outputCol="pca_features")
pca_model = pca.fit(content_df)
pca_df = pca_model.transform(content_df)

print(f"Variance expliquée par 2 composantes PCA: {sum(pca_model.explainedVariance):.2%}")

# Extraire les coordonnées x, y pour chaque event (sample de 5000 pour la visu)
pca_sample = pca_df.select(
    F.col("content_cluster"),
    F.col("platform"),
    pca_df["pca_features"][0].alias("pca_x"),
    pca_df["pca_features"][1].alias("pca_y")
).sample(False, 0.02, seed=42).limit(5000)

pca_out = os.path.join(WAREHOUSE, "pca_viz")
pca_sample.write.mode("overwrite").parquet(pca_out)

print(f"Données PCA écrites dans : {pca_out}")
pca_sample.show(5)

In [ ]:
# ── D2. VISUALISATION MATPLOTLIB (optionnel, pour le notebook) ────────────────
try:
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm
    import numpy as np

    pdf = pca_sample.toPandas()

    colors = cm.tab10(np.linspace(0, 1, K_CONTENT))
    fig, ax = plt.subplots(figsize=(10, 8))

    for cid in range(K_CONTENT):
        mask = pdf["content_cluster"] == cid
        label = CONTENT_LABELS.get(cid, {}).get("label", f"C{cid}")
        ax.scatter(pdf[mask]["pca_x"], pdf[mask]["pca_y"],
                   c=[colors[cid]], label=label, alpha=0.4, s=8)

    ax.set_title("MyDigitalTwin — Galaxie des centres d'intérêts (PCA 2D)", fontsize=14)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    plt.tight_layout()
    plt.savefig(os.path.join(WAREHOUSE, "../../../scripts/notebooks", "app", "assets", "pca_galaxy.png"), dpi=120)
    plt.show()
    print("Galaxie sauvegardée dans app/assets/pca_galaxy.png")
except ImportError:
    print("matplotlib non disponible — skipping visualization")

---
## Résumé des outputs

| Table | Contenu | Utilisée par |
|---|---|---|
| `warehouse/content_clusters` | 8 clusters TF-IDF avec top_keywords | home.py (orbit tags) |
| `warehouse/behavioral_clusters` | 6 profils temporels | home.py (section profils) |
| `warehouse/interest_profiles` | Fusion : label hybride + keywords + heure | home.py (remplacement CATEGORY_KEYWORDS) |
| `warehouse/pca_viz` | Coordonnées 2D PCA (5k points) | Dashboard visualisation |

**Prochaine étape** : mettre à jour `app/pages/home.py` pour lire `interest_profiles` au lieu du dict `CATEGORY_KEYWORDS` codé en dur.

In [ ]:
spark.stop()
print("Spark session fermée. Notebook terminé.")